# Lab 04 — Narrow Transformations

**Tujuan:** Mengamati filter, select, dan withColumn tanpa redistribusi wajib.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 04")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
df = spark.range(0, 2_000_000, 1, 4)
filtered = df.filter("id < 100000")
changed = filtered.withColumn("double_id", F.col("id") * 2)
result = changed.select("id", "double_id")
result.explain("formatted")

**Prediksi:** apakah ada `Exchange`? Apakah semua transformasi menambah stage baru?

In [ ]:
result.count()

## Compare with repartition

Tambahkan operasi yang memindahkan record antarp­artisi.

In [ ]:
moved = result.repartition(8)
moved.explain("formatted")
moved.count()

**Tugas:** inspeksi tab Stages dan perbedaan shuffle read/write.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.